# Aralin 18: Pag-secure ng mga AI Agent gamit ang Cryptographic Receipts

## Hands-on Notebook

Tinuturuan ka ng notebook na ito sa apat na pagsasanay:

1. **Pirmahan ang iyong unang resibo** para sa tawag ng agent tool at i-verify ito.
2. **Bungkagin ang resibo** at panoorin ang pagpapatunay na pumalya.
3. **Gumawa ng tatlong-resibo na chain** at kumpirmahin ang integridad ng chain.
4. **Balutin ang tawag sa Microsoft Agent Framework tool** para bawat aksyon ay maglabas ng isang resibo.

Lahat ng cryptographic primitives ay inangkat mula sa mga maayos na pinangangalagaang mga library (`pynacl` para sa Ed25519, `jcs` para sa RFC 8785 canonical JSON, `hashlib` mula sa Python standard library para sa SHA-256). Ang logic ng resibo mismo ay simpleng Python na maaari mong basahin at baguhin.

Patakbuhin ang mga cell nang sunod-sunod. Bawat seksyon ay maikli at nakahiwalay.


## Setup

I-install ang dalawang dependencies. Pareho silang may mga permissive na lisensya (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Mga Tulong na Utility

Pinangangasiwaan ng dalawang tulong na ito ang base64url encoding (nang walang padding) at SHA-256 hashing ng anumang mga bagay. Pinananatili nila ang natitirang bahagi ng notebook na nakatuon sa mismong lohika ng resibo.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Seksyon 1: Lagdaan ang iyong unang resibo

Isipin na lang na ang aming ahente para sa **Contoso Travel** ay naghanap lang ng mga flight mula Sydney papuntang Los Angeles para sa isang customer. Nais naming irekord ang pahayag ng gateway tungkol sa tawag ng tool na ito upang ma-verify ito ng isang auditor sa hinaharap sa ilalim ng hiwalay na pinned na issuer key.

### Hakbang 1.1: Bumuo ng isang signing key

Sa production, ang signing key ng ahente ay ilalagay sa hardware security module (HSM), Azure Key Vault, o katulad na protektadong imbakan. Para sa araling ito, gagawa tayo ng bagong key sa memorya.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Hakbang 1.2: Buuhin ang payload ng resibo

Ang payload ay naglalaman ng lahat ng gusto nating patunayan ng resibo: sino ang kumilos, anong kasangkapan, gamit ang anong mga argumento, ano ang bumalik, sa ilalim ng anong patakaran, at kailan. Hini-hash namin ang mga argumento at resulta sa halip na isama ang mga ito nang inline upang hindi magtagas ang resibo ng sensitibong nilalaman.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Hakbang 1.3: Pirmahan at buuin ang resibo

Tatlong hakbang:

1. Canonicalize ang payload gamit ang JCS upang ang dalawang implementasyon na gumagawa ng parehong lohikal na resibo ay makabuo ng byte-identical na bytes.
2. Pirmahan nang direkta ang canonical bytes gamit ang Ed25519 private key. Ang PureEdDSA ay nagha-hash ng mensahe nang internal, kaya ang karagdagang pre-hash ay magbabago sa protocol.

Ang pirma ay ikinakabit sa orihinal na payload upang mabuo ang pinal na resibo.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Hakbang 1.4: Suriin ang resibo

Ang pag-verify ay pinapal reversing ang proseso. Inaalis namin ang pirma, muling kinakalkula ang canonical bytes, at sinusuri ang pirma laban sa pampublikong susi ng issuer.

Ang pag-verify ay offline: walang serbisyong tatawagan at walang access sa network. Kailangan pa rin nito ng isang bagay bukod sa resibo: ang pampublikong susi ng issuer, na nakuha sa pamamagitan ng channel na pinagkakatiwalaan na ng auditor, tulad ng JWK Set na inilathala ng issuer o isang halagang na-pin nang maaga. Ang `public_key` na patlang sa loob ng resibo ay hindi maaaring gampanan ang papel na iyon, dahil ang sinumang sumusulat ng resibo ay siya ring sumusulat ng patlang na iyon. Ipinapakita ng Seksyon 2 ang peke na ito na napipigilan.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Dapat mong makita ang `Receipt is valid: True`, `Pre-hashed receipt valid: False` at `Malformed receipts rejected: True`. Pinapatunayan ng positibong kaso na gumagana ang direktang-JCS na landas ng pagpirma; ang negatibong kontrol ay ginagawang executable ang alituntunin ng saklaw ng pirma imbes na iwan ito bilang prosa; ang huling linya ay nagpapakita ng maling format ng input na nagbabalik ng `False` sa halip na magtaas ng eksepsyon.


## Seksyon 2: Panghimasok sa resibo

Ang buong layunin ng mga resibo ay para makita ang anumang panghimasok. Patunayan natin ito.

Babaguhin natin ang eksaktong isang karakter ng resibo at panoorin ang pagkabigo ng beripikasyon.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Ano ang nangyari?

Nang binago namin ang `policy_id`, nagbago ang canonical bytes. Ang lagda (na nasa orihinal na canonical bytes) ay hindi na tumutugma. Tama ang pagbabalik ng beripikasyon ng `False`.

Wala nang paraan para baguhin ang isang nilagdaang field, ibig sabihin anumang bagay na nasa labas ng `signature` na object, at magpatunay pa rin ang resibo gamit ang pampublikong susi ng nag-isyu, maliban kung hawak ng umaatake ang pribadong susi ng nag-isyu. Ang `signature` na object mismo ay hindi nilagdaan: ang `sig` ang lagda, at ang `verify_receipt` ay hindi pinapansin ang `public_key`, kaya ang pag-edit lamang ng field na iyon ay walang epekto. Hangga't ang pribadong susi ay nasa key vault at gumagamit ang mga tagapagberipika ng independently na inilathalang susi ng nag-isyu, imposibleng maitago ang pagbabago sa isang nilagdaang field. Mahalaga ang kondisyong iyon: ipinapakita ng susunod na cell ang binagong resibo na perpektong nagve-verify laban sa ibang susi.

Subukan mo mismo: baguhin ang `tool_name` o `agent_id` o `timestamp` sa cell sa itaas at patakbuhin muli. Bawat pagbabago sa isang nilagdaang field ay nagreresulta sa hindi wastong resibo.


### Ang panlilinlang na hindi nahuhuli ng pirma

Ang umaatake sa itaas ay hindi muling pumirma. Ang mas maingat ay: babaguhin ang resibo,
pipirmahan ito gamit ang isang susi na siya mismo ang gumawa, at ilalagay ang susi na iyon sa
`public_key` na patlang. Ang resulta ay isang resibo na ang pirma ay ganap na tunay
para sa nilalaman nito.

Patakbuhin ito at tingnan kung alin sa dalawang pagsusuri ang humuhuli nito.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Dapat mong makita ang `True` pagkatapos ang `False`. Ang tatlong linya ng `assert` ay ginagawa itong isang
tseke: ang cell ay nagtaas ng error kung ang pamemeke ay kailanman mare-verify laban sa
susi ng issuer.

Ang pamemeke ay internally consistent. Ang lagda nito ay talagang tumutugma sa
nilalaman at susi na dala nito, kaya ang isang verifier na nagbabasa ng susi mula sa
resibo ay nag-uulat nito bilang wasto. Wala sa resibo ang mukhang mali, dahil
wala naman talagang inconsistency dito.

Ang tanging nakakahuli nito ay ang pagtseke laban sa susi na walang paraan na naimpluwensyahan ng attacker. Ito ang dahilan kung bakit
tinatanggap ng `verify_receipt` ang susi bilang argumento: ang
isang function na may lagda na `verify_receipt(receipt)` ay hindi maaaring isulat nang ligtas,
dahil ang tanging susi na maabot nito ay ang isa na ibinigay ng attacker.

Ang parehong resibo ay nase-save bilang `sample_receipts/04_resigned_receipt.json` kung nais mong
suriin ito sa labas ng notebook.


## Seksyon 3: Magkadugtong na mga resibo

Ang isang resibo ay nagpoprotekta sa isang aksyon. Karamihan sa mga ahente ay gumagawa ng maraming aksyon. Upang gawing tamper-evident ang buong pagkakasunod-sunod, ginagawang konektado ang bawat resibo sa naunang isa sa pamamagitan ng pagsama ng hash ng naunang resibo sa payload ng bagong resibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kung may magtanggal o mag-ayos muli ng resibo sa loob ng ipinakitang pagkakasunod-sunod, masisira ang susunod na link. Ang pagtuklas ng pagbabawas o pagsulat muli ng sumunod na bahagi na kontrolado ng tagapirma ay nangangailangan din ng naka-pin na kasalukuyang ulo, panlabas na checkpoint, o sertipikadong pangako ng populasyon.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Ngayon sirain ang kadena sa pamamagitan ng pakikialam sa gitnang resibo at muling i-verify. Nabigo ang napanghimasok na resibo sa pagsusuri ng lagda nito, AT ang susunod na resibo ay nabigo sa pagsusuri ng pagkakaugnay ng kadena (dahil ang `previous_receipt_hash` nito ay hindi na tumutugma sa binagong hash ng gitnang resibo).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Tinatanggap pa rin ang Resibo 0 (hindi ito nabago at walang naunang kinalalagyan na pagdedependehan). Nabigo ang Resibo 1 sa pag-verify ng lagda nito dahil binago namin ang `tool_args_hash`. Nabigo ang Resibo 2 sa pag-verify ng chain-link nito dahil ang `previous_receipt_hash` nito ay kinompyut laban sa orihinal (na ngayo'y nabago) na resibo 1.

Kahit na muling pirmahan ng attacker ang nabagong resibo 1, gagalaw ang bagong lagda laban sa susi ng nag-isyu maliban na lang kung hawak nila ang pribadong susi ng nag-isyu (ang muling paglagda gamit ang sariling susi ay nabigo, gaya ng ipinakita sa Seksyon 2), at ipapakita pa rin ng chain-link na hindi tumutugma sa resibo 2 ang panlilinlang. Para itago ang pagbabago, kailangan muling pirmahan ng attacker ang bawat resibo mula sa puntong binago pasulong, na nangangailangan ng pagmamay-ari ng pribadong susi ng nag-isyu.


## Seksyon 4: Balutin ang pagtawag sa agent tool gamit ang pagpirma ng resibo

Sa isang totoong deployment, ayaw mong alalahanin ng bawat manunulat ng agent na tawagan ang `make_receipt`. Gusto mong maging awtomatiko ang pagpirma ng resibo para sa bawat pagtawag sa tool.

Narito ang pinakasimpleng pattern: isang wrapper class na kumukuha ng anumang callable tool function at nagbabalik ng bersyon nito na naglalabas ng resibo. Ito ay umaangkop sa anumang agent framework, kasama na ang Microsoft Agent Framework (`agent_framework.foundry`).

Kung wala ka pang Microsoft Foundry project na naka-setup, ipinapakita pa rin ng lokal na mock sa ibaba ang pattern.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Pagsasama sa Microsoft Agent Framework

Ang `ReceiptedTool` na pambalot sa itaas ay hindi nakadepende sa framework. Upang magamit ito sa loob ng isang ahente na ginawa gamit ang Microsoft Agent Framework, irehistro ang balot na function bilang isang tool. Isang sketch (papalitan mo ang mock ng isang tunay na pagpaparehistro ng tool sa Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Hindi kailangang malaman ng agent framework ang anuman tungkol sa mga resibo. Ang pagpirma sa resibo ay nakabalot sa paligid ng tool, hindi nakatali sa framework. Ganito mo idinadagdag ang pinagmulan sa umiiral na code ng ahente nang hindi muling sinusulat ang ahente.


## Buod at hamon sa pag-unlad

Nagawa mo:

- Nakabuo ng Ed25519 na pares ng susi.
- Nakagawa at pumirma ng isang resibo para sa tawag sa tool ng ahente.
- Napatunayan ang resibo nang offline gamit lamang ang pampublikong susi ng issuer, na nakuha nang hiwalay mula sa resibo.
- Pinagbago ang isang resibo at napansin ang kabiguan sa pagpapatunay.
- Nakabuo ng hash-chained na sunod-sunod ng tatlong mga resibo.
- Pinagbago ang gitna ng chain at napansin ang parehong kabiguan sa pirma at kabiguan sa link ng chain.
- Nilagyan ng awtomatikong paglagda ng resibo ang isang function ng tool.

**Hamon sa pag-unlad.** Palawakin ang schema ng resibo gamit ang isang `request_id` na field (isang UUID para sa distributed tracing). I-update ang `make_receipt` upang isama ito, at kumpirmahing patuloy na napatunayan ang mga resibo nang buo. Pagkatapos, baguhin ang field pagkatapos ng paglagda at kumpirmahing mabibigo ang pagpapatunay. Pinipilit ka nitong intindihin kung paano nakakatulong ang bawat byte ng canonical encoding sa pirma.

**Mahalagang hangganan.** Sa ilalim ng pin na susi at konteksto ng chain, pinatutunayan ng mga resibo ang attribution ng pirma, integridad ng nilalaman, at isang link sa ipinakitang naunang bahagi. Hindi nito pinatutunayan ang pagpapatupad, katotohanan ng pinagmulan, pinagkakatiwalaang oras, pagkakumpleto, pagsusuri ng polisiya, o kawastuhan. Ang mga resibo ay pundasyon. Ang pamamahala ang sistemang iyong itinatayo sa ibabaw nito.

Basahin muli ang README ng aralin na may isinaalang-alang na hangganan. Ang pinakakaraniwang pagkakamali ng mga koponan sa mga resibo ay ang pag-aakala na "mayroon tayong mga resibo" ay nangangahulugan ng "tayo ay napamamahalaan." Hindi ito totoo. Ginagawa ng mga resibo na ma-audit ang kilos ng ahente. Hindi nito ginagawang tama ang mga ito.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
